# CheXpert Active Labeling Inference

Cleaned batch odds-ratio workflow for the paper example. Original exploratory notebooks are preserved in `archive/`.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

import numpy as np
import pandas as pd

CWD = Path.cwd().resolve()
REPO_ROOT = CWD.parent if CWD.name in {'Stance', 'Alphafold', 'CheXpert', 'BRCA'} else CWD
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from utils import (
    binary_odds_ratio_truth,
    HUMAN_N_COL,
    EFFECTIVE_N_COL,
    run_odds_ratio_monte_carlo,
    summarize_monte_carlo,
)
from plotting import (
    make_monte_carlo_variance_table,
    plot_coverage,
    plot_effective_sample_size,
    plot_finite_population_coverage,
    plot_intervals,
    plot_monte_carlo_variance,
    plot_monte_carlo_variance_components,
    save_monte_carlo_variance_table,
)


In [ ]:
EXAMPLE_DIR = REPO_ROOT / "CheXpert"
DATA_DIR = REPO_ROOT / "Data" / "CheXpert"
PLOTS_DIR = EXAMPLE_DIR / "plots"
RESULTS_DIR = EXAMPLE_DIR / "results"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SEED = 614
ALPHA = 0.1
TAU = 0.5
FRACS_HUMAN = np.linspace(0.1, 0.2, 20)
NUM_TRIALS = 100


In [ ]:
master_df = pd.read_csv(DATA_DIR / "chex_chexmodel_withpreds_AP_PA.csv")

OUTCOME_COL = "Cardiomegaly"
PRED_COL = "prob_Cardiomegaly"
GROUP_COL = "AP/PA"

analysis_df = master_df.copy()
analysis_df[OUTCOME_COL] = pd.to_numeric(analysis_df[OUTCOME_COL], errors="coerce")
analysis_df = analysis_df[
    analysis_df[OUTCOME_COL].isin([0, 1])
    & analysis_df[PRED_COL].notna()
    & analysis_df[GROUP_COL].isin(["AP", "PA"])
].copy()
analysis_df["AP_0/1"] = (analysis_df[GROUP_COL] == "AP").astype(int)

Y = analysis_df[OUTCOME_COL].astype(int).to_numpy()
Yhat = analysis_df[PRED_COL].astype(float).to_numpy()
AP = analysis_df["AP_0/1"].to_numpy(dtype=bool)

Y0, Yhat0 = Y[~AP], Yhat[~AP]
Y1, Yhat1 = Y[AP], Yhat[AP]
true_odds_ratio, true_variance = binary_odds_ratio_truth(Y0, Y1)
mu0_pilot, mu1_pilot = float(np.mean(Yhat0)), float(np.mean(Yhat1))

pd.DataFrame(
    {
        "group": ["PA", "AP"],
        "n": [len(Y0), len(Y1)],
        "outcome_mean": [Y0.mean(), Y1.mean()],
        "prediction_mean": [Yhat0.mean(), Yhat1.mean()],
    }
)


In [ ]:
df = run_odds_ratio_monte_carlo(
    y0=Y0,
    yhat0=Yhat0,
    y1=Y1,
    yhat1=Yhat1,
    fracs_human=FRACS_HUMAN,
    alpha=ALPHA,
    num_trials=NUM_TRIALS,
    true_odds_ratio=true_odds_ratio,
    true_variance=true_variance,
    mu0_pilot=mu0_pilot,
    mu1_pilot=mu1_pilot,
    tau=TAU,
    seed=SEED,
    split_spline_budget_evenly=False,
    show_progress=True,
)

summary_df = summarize_monte_carlo(df)
mc_variance_table = make_monte_carlo_variance_table(df)

df.to_csv(RESULTS_DIR / "CheXpert_results.csv", index=False)
summary_df.to_csv(RESULTS_DIR / "CheXpert_monte_carlo_summary.csv", index=False)
mc_variance_table.to_csv(RESULTS_DIR / "CheXpert_monte_carlo_variance_components.csv", index=False)

mc_variance_table.head(12)


In [ ]:
n_total = len(Y0) + len(Y1)
plot_effective_sample_size(
    df,
    path=PLOTS_DIR / "CheXpert_effective_sample_size.png",
    n_total=n_total,
    error_bars="sd",
    show=False,
)
plot_effective_sample_size(
    df,
    path=PLOTS_DIR / "CheXpert_effective_sample_size_no_error_bars.png",
    n_total=n_total,
    error_bars="none",
    show=False,
)
plot_coverage(
    df,
    alpha=ALPHA,
    path=PLOTS_DIR / "CheXpert_coverage.png",
    n_total=n_total,
    show=False,
)
plot_finite_population_coverage(
    df,
    alpha=ALPHA,
    path=PLOTS_DIR / "CheXpert_coverage_finite_population_calibrated.png",
    n_total=n_total,
    show=False,
)
plot_monte_carlo_variance(
    df,
    path=PLOTS_DIR / "CheXpert_monte_carlo_variance.png",
    n_total=n_total,
    show=False,
)
plot_monte_carlo_variance_components(
    df,
    path=PLOTS_DIR / "CheXpert_monte_carlo_variance_components.png",
    n_total=n_total,
    show=False,
)
save_monte_carlo_variance_table(
    df,
    path=PLOTS_DIR / "CheXpert_monte_carlo_variance_table.png",
    max_rows=18,
    show=False,
)
plot_intervals(
    df,
    true_value=true_odds_ratio,
    path=PLOTS_DIR / "CheXpert_intervals.png",
    estimand_label="odds ratio: AP vs PA",
    show=False,
)
